# 🚀 Project Blueprint: NLP Search & Recommendation System

> **Mission Objective:** Build a lightweight search engine that matches a user's natural language query against a document corpus, ranking the most relevant results using **TF-IDF representation** and **Cosine Similarity metrics**.

---

## 🎬 Example Scenario

* **The Corpus:** You have a database of 10 movie descriptions.
* **The Search Query:** `“space adventure with robots”`
* **The Expected Output:** The system instantly evaluates all 10 descriptions and serves the top movies mathematically closest to that theme.

---

## 🛠️ Step-by-Step Implementation Pipeline

### Phase 1: Dataset Construction
* [ ] **Minimum Baseline:** Create a collection of at least **10 short text documents**.
* [ ] **Data Schema:** Ensure each record contains exactly two fields:
  1. `Title` (e.g., *Interstellar*)
  2. `Description` (e.g., *A team of explorers travel through a wormhole in space...*)

### Phase 2: Text Preprocessing Pipeline
To keep our vectors clean, both the dataset and the incoming user queries must pass through the exact same cleaning station:

| Step | Operation | Purpose |
| :---: | :--- | :--- |
| **1** | **Lowercase conversion** | Standardizes text so `Space` and `space` match perfectly. |
| **2** | **Punctuation Stripping** | Removes commas, periods, and exclamation marks (`!`, `.`, `,`). |
| **3** | **Stopwords Exclusion** | Drops low-value filler words like `is`, `a`, `and`, `the`. |
| **4** | **Normalization** | Applies **Lemmatization** or **Stemming** to reduce words to their base form (e.g., `robots` ➡️ `robot`). |

### Phase 3: Mathematical Modeling & Matching
* [ ] **Vectorization:** Convert the fully preprocessed text documents into numeric matrices using `scikit-learn`'s **`TfidfVectorizer`**.
* [ ] **Query Handling:** Capture a runtime user query (e.g., `"healthy food"` or `"space adventure"`), subject it to the Phase 2 Preprocessing workflow, and project it into the same TF-IDF space.
* [ ] **Similarity Calculation:** Run a **Cosine Similarity** computation comparing the standalone query vector against every single document vector in the corpus.

---

## 🎯 Final Output Specifications

Your script must sort the mathematical alignment scores in descending order and return exactly the **Top 3 Results** matching the user's intent. Display each result on the terminal using the following visual template:

```text
==================================================
🏆 RANK [1/2/3] | SIMILARITY SCORE: [0.XX]
==================================================
Title:       [Movie Title Here]
Description: [Movie Plot Summary Here]
==================================================

Load dataset

↓

Inspect dataset

↓

Clean messy values

↓

Create combined_text

↓

Preprocess text

↓

Create TF-IDF matrix

↓

Accept user query

↓

Preprocess query

↓

Convert query to TF-IDF vector

↓

Calculate cosine similarity

↓

Return top 3 movies

## Phase 1 : Load the dataset and Understand it

In [150]:
from pathlib import Path
import pandas as pd
path=Path("../data/raw/messy_real_movies_nlp_550-1.csv")
df=pd.read_csv(path)
print(f"{df.shape}\n")
print(df.head(2))
df.columns.tolist()

(550, 12)

   movie_id             title  \
0     10061  Escape from L.A.   
1    239571    The Best of Me   

                                         description           genre  \
0  This time, a cataclysmic temblor hits Los Ange...             NaN   
1  A pair of former high school sweethearts reuni...  Drama, Romance   

                                              review  \
0   not bad not great; audience_score:5.6; votes:373   
1  Mixed-to-positive audience reaction. Some view...   

                                              actors        directors  \
0  Kurt Russell | Stacy Keach | Steve Buscemi | P...   John Carpenter   
1  Michelle Monaghan, James Marsden, Liana Libera...  Michael Hoffman   

  release_date original_language vote_average  vote_count  \
0   09-08-1996                en          5.6         373   
1   16-10-2014                en          7.2         765   

            source_dataset  
0  TMDB 5000 Movie Dataset  
1  TMDB 5000 Movie Dataset  


['movie_id',
 'title',
 'description',
 'genre',
 'review',
 'actors',
 'directors',
 'release_date',
 'original_language',
 'vote_average',
 'vote_count',
 'source_dataset']

In [151]:
df.isnull().sum()

movie_id              0
title                 0
description          15
genre                13
review               14
actors                8
directors            14
release_date          8
original_language     0
vote_average          0
vote_count            0
source_dataset        0
dtype: int64

# DATA CLEANING ALERT: Missing Value Audit Dashboard

> 📉 **Current Status:** Your dataset has holes that will make your machine learning models or vectorizers crash if left untreated. Here is your battlefield breakdown.

| 📋 Column Name | ❌ Missing Count | 📊 Severity Level | ⚡ Risk / Impact on NLP |
| :--- | :---: | :--- | :--- |
| **`description`** | **15** | 🟥 **CRITICAL** (Highest) | **Fatal.** `TfidfVectorizer` and `CountVectorizer` will completely crash if they hit a `NaN` value here. |
| **`review`** | **14** | 🟥 **CRITICAL** | **High Loss.** Ruining your conditional group-mean calculations if left un-imputed. |
| **`directors`** | **14** | 🟧 **HIGH** | Breaks feature engineering if you try to combine metadata strings later. |
| **`genre`** | **13** | 🟧 **HIGH** | Strips away primary filtering power for the recommendation engine. |
| **`actors`** | **8** | 🟨 **MEDIUM** | Moderate structural loss; easier to patch with string placeholders. |
| **`release_date`** | **8** | 🟨 **MEDIUM** | Time-based features will have gaps, causing sorting issues. |

## Phase 2 : Create working dataframe

Keep all columns, but start focusing on these:

title

description

genre

actors

vote_average

review

In [152]:
# Make a copy first:-
df_copy=df.copy()
# Defining the columns to focus on:-
focus_columns = [
    "title",
    "description",
    "genre",
    "actors",
    "vote_average",
    "review",
    "directors",
    "release_date"
]
# Checking missing values in the focus columns:-
na_counts = df_copy[focus_columns].isna().sum()
datatypes = df_copy[focus_columns].dtypes
na_counts=df_copy[focus_columns].isna().sum()
print("Missing Values Count in Focus Columns:")
tab= pd.DataFrame({
    "Column": focus_columns,
    "Missing Values": na_counts,
    "Data Type": datatypes,
    "NA Count": na_counts
}).reset_index(drop=True)
print(tab)



Missing Values Count in Focus Columns:
         Column  Missing Values Data Type  NA Count
0         title               0       str         0
1   description              15       str        15
2         genre              13       str        13
3        actors               8       str         8
4  vote_average               0       str         0
5        review              14       str        14
6     directors              14       str        14
7  release_date               8       str         8


## Phase 3 — Create the combined search text column

Phase 3, we will combine selected text columns into one column called combined_text

In this phase, we combine important text columns into one column called `combined_text`.

The TF-IDF system needs one text document per movie.

We combine:
- description
- genre
- actors
- review
- directors

We do not include `vote_average` yet because it is numeric.
We do not include `release_date` yet because it is date information.
These columns will be shown later in the final result table.

In [153]:
categorical_columns = [
    "description",
    "genre",
    "actors",
    "review",
    "directors"
]
# Replacing missing values in categorical columns with "" or emptyness:-
df_copy[categorical_columns]=df_copy[categorical_columns].fillna("")

- We are not deleting rows.
- We are only making missing text safe.

In [154]:
df_copy["combined_text"]=(df_copy["description"]+" "+df_copy["genre"]+" "+df_copy["actors"]+" "+df_copy["review"]+" "+df_copy["directors"])

## ⛓️ Pandas Text Concatenation Pipeline



| Code Component | Meaning / Role | Behind-the-Scenes Action |
| :--- | :--- | :--- |
| **`movies_df["combined_text"]`** | Creates a new column | Tells pandas to allocate memory for a brand new column named `combined_text` to store our final target strings. |
| **`movies_df["description"]`** | Pulls base text | Acts as the starting point, grabbing the raw text array from the original `description` column. |
| **`+ " "`** | Adds a space | Injects a literal blank space string between columns so the words don't smash together (e.g., preventing `"Sci-FiAction"` and making it `"Sci-Fi Action"`). |
| **`.astype(str)`** | Data type safeguard | Forces every single element in that column to become an explicit string. This prevents your script from crashing if it hits numbers or hidden `NaN` values. |
| **Final Result** | One big searchable column | Combines descriptions, genres, and metadata into a single massive string per row, ready to be fed directly into your `TfidfVectorizer`. |

In [155]:
df_copy[["title","combined_text"]].head(2)

,title,combined_text
0,Escape from L.A.,"This time, a cataclysmic temblor hits Los Ange..."
1,The Best of Me,A pair of former high school sweethearts reuni...


## 🎬 Feature Engineering: Building the Movie Profile Matrix



| Target Column | Source Data Components | NLP Role & Core Purpose |
| :--- | :--- | :--- |
| **`title`** | Movie Title | **The Label:** The human-readable identity of the film. Your model does *not* look at this to find similarities; it is purely used to display the final top-3 list back to the user. |
| **`combined_text`** | `description` + `genre` + `actors` + `review` + `directors` | **The Master Profile (The Soup):** Smashes all textual and categorical features into a single massive string per movie. This gives your `TfidfVectorizer` full contextual awareness of the plot, themes, cast, and creative style all at once. |

## Phase 4: Preprocess the Combined Text

In this phase, we clean the text before applying TF-IDF.

We will:
- convert text to lowercase
- remove punctuation using the `re` library
- remove stopwords using spaCy
- lemmatize words using spaCy

We use the same preprocessing function later for both:
- movie text
- user search query

In [156]:
import re
import spacy

### The following function will:-

## ⚙️ The Ultimate Text Preprocessing Pipeline Breakdown



> **System Blueprint:** This function serves as an assembly line that takes raw, messy human language and refines it into clean, standardized tokens that a machine learning model can actually understand.

| Step | Code Snippet | What it is Called | 🧠 Mental Note (The "Why") |
| :---: | :--- | :--- | :--- |
| **1** | `nlp = spacy.load("en_core_web_sm")` | **Load spaCy Model** | Wakes up the English NLP engine and loads up its built-in dictionaries. |
| **2** | `def preprocessText(text):` | **Function Definition** | Builds a reusable, automated cleaning machine. |
| **3** | `text = str(text)` | **Type Conversion** | Force-stops crashes by converting any input data type cleanly into a string. |
| **4** | `text = text.lower()` | **Lowercasing** | Standardizes cases. Makes `Robot`, `ROBOT`, and `robot` look identical to the machine. |
| **5** | `re.sub(r"[^a-z\s]", " ", text)` | **Regex Cleaning** | **The Eraser:** Obliterates numbers, emojis, and punctuation, leaving *only* basic letters and spaces. |
| **6** | `re.sub(r"\s+", " ", text).strip()` | **Whitespace Normalization** | Squeezes multiple ugly spaces down into a single clean space, then trims the edges. |
| **7** | `doc = nlp(text)` | **spaCy Processing** | Tokenizes the text, turning a raw string into an intelligent, analyzed `spaCy Doc` object. |
| **8** | `clean_words = []` | **Empty List Creation** | Hooks up an empty storage basket to catch your finalized, clean words. |
| **9** | `for i in doc:` | **Looping** | Grabs a magnifying glass to check every single word (token) one by one. |
| **10** | `if not i.is_stop and not i.is_punct:` | **Filtering** | **The Bouncer:** Kicks out useless filler words (stop words like `is`, `the`) and punctuation. |
| **11** | `clean_words.append(i.lemma_)` | **Lemmatization + Append** | Strips words down to their root form (e.g., `running` ➡️ `run`) and drops them in the basket. |
| **12** | `clean_text = " ".join(clean_words)` | **Joining** | Glues the isolated list of clean words back together into a normal, readable sentence. |
| **13** | `return clean_text` | **Return Statement** | Ships the finalized, sparkling clean text string right out of the function! |

In [ ]:
nlp=spacy.load("en_core_web_sm")
def preprocessText(text):
    text=str(text)
    text=text.lower()
    text=re.sub(r"[^a-z\s]", " ", text)
 # Syntax:- re.sub(pattern, replacement, string)
# [] defines character group.
# ^ inside [] means negation or not.
# a-z means all lowercase letters, 
# \s means whitespace characters.
# Find anything that is NOT a lowercase letter and NOT a space.
# Replace those characters with a space " ".
# In the following step, we will replace multiple 
# spaces with a single space and also remove leading and trailing spaces:-
    text=re.sub(r"\s+", " ", text).strip()
    doc=nlp(text) # Process the text with spaCy to create a Doc object.
    clean_words=[]
    # Iterate through each token in the Doc object:-
    for i in doc:
        if not i.is_stop and not i.is_punct:  # Agar token stop word nahi 
            #hai aur punctuation mark nahi hai, tabhi usko aage condition k liye bhejenge:-

            #is_stop :- ka kaam hai ki wo 
            # check karta hai ki token ek stop 
            # word hai ya nahi, jaise "the", "is", 
            # "in", etc., aur unko clean words list se exclude karta hai.

            # is_punct :- ka kaam hai ki wo check karta hai ki token 
            # ek punctuation mark hai ya nahi, jaise ".", ",", "!", 
            # etc., aur unko bhi clean words list se exclude karta hai.

            # Toh yahaan humne if condition k andar check kiya hai ki agar 
            # token stop word nahi hai aur punctuation mark nahi hai, 
            # tabhi usko clean words list me add karenge:-

            clean_words.append(i.lemma_) #i.lemma_ ka kaam hai ki wo token ka lemmatized 
            #form return karta hai.
            #For example, "running" would be lemmatized to "run", and "better" 
            # would be lemmatized to "good".

    clean_text=" ".join(clean_words) #Yahaan maine clean_words list me jo 
    #bhi words add kiye hain, unko space se join karke ek single string 
    # me convert kar diya hai, jise clean_text variable me store kiya hai.
    
    return clean_text

- *** Text preprocessing pipeline explained ***

>> Takes raw movie text, cleans it using regex, removes stopwords using spaCy, lemmatizes the remaining words, and returns one clean text string for TF-IDF.

>> String → Lowercase → Regex Clean → Regex Spaces → spaCy Doc → Empty List → Loop Tokens → Remove Stopwords/Punctuation → Lemmatize → Join → Return

>> S L R R D L F L J R (Remember it like this)

## Phase 5 — Text Preprocessing

1. lowercase
2. remove punctuation
3. remove stopwords
4. lemmatize or stem words

## Phase 6 — Convert Text into TF-IDF Vectors

## Phase 7 — Accept a User Query

## Phase 8 — Convert the Query into a TF-IDF Vector

## Phase 9 — Calculate Cosine Similarity

## Phase 10 — Return Top 3 Results

## Phase 11 — Test with Multiple Queries

## Phase 12 — Write Final Explanation / Report Section